In [ ]:
import os, random, shutil
from pathlib import Path
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
CONFIG = {
    "work_base": "/kaggle/working/poaching-dataset",
    "test_src": {
        "animal": "/kaggle/input/dataset1/animal-1/WAID-final - Copy/images/test",
        "empty": "/kaggle/input/dataset1/human-1/human detection dataset/0"
    },
    "human_cutouts": "/kaggle/input/h1cutouts/human-cutouts",
    "wildlife_bg": "/kaggle/input/h1cutouts/wildlife-backgrounds",
    "img_size": (320, 320),
    "batch_size": 16,
    "seed": 42,
    "synthetic_test_count": 200
}

# Set seeds for reproducibility
np.random.seed(CONFIG["seed"])
random.seed(CONFIG["seed"])
tf.random.set_seed(CONFIG["seed"])

print("--- CONFIGURATION ---")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print("---------------------")

# --- SETUP TEST DIRECTORIES ---
test_dir = Path(CONFIG["work_base"]) / "test"
print(f"Setting up test directory: {test_dir}")
for cls in ["animal", "empty", "poacher"]:
    os.makedirs(test_dir/cls, exist_ok=True)
print("✅ Test directories created.")

# --- COPY REAL TEST IMAGES ---
def copy_images(src, dst):
    """Copies image files from source to destination."""
    os.makedirs(dst, exist_ok=True)
    count = 0
    for p in Path(src).glob("*"):
        # Check for common image file extensions
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            shutil.copy(p, dst)
            count += 1
    return count

print("\n--- COPYING REAL TEST IMAGES ---")
animal_count = copy_images(CONFIG["test_src"]["animal"], test_dir/"animal")
print(f"Copied {animal_count} real 'animal' images.")
empty_count = copy_images(CONFIG["test_src"]["empty"], test_dir/"empty")
print(f"Copied {empty_count} real 'empty' images.")

# --- SYNTHETIC IMAGE GENERATION FOR TEST SET (POACHER + EMPTY) ---
print("\n--- PREPARING SYNTHETIC DATA GENERATION ASSETS ---")
# Get lists of human cutouts and wildlife backgrounds
humans = list(Path(CONFIG["human_cutouts"]).glob("*.png"))
bgs = [p for p in Path(CONFIG["wildlife_bg"]).glob("*") if p.suffix.lower() in [".jpg", ".png"]]

# Split backgrounds for poacher and empty synthetic images
random.shuffle(bgs) # Ensure the split is random
poacher_bgs = random.sample(bgs, int(len(bgs)*0.6))
empty_bgs = [b for b in bgs if b not in poacher_bgs]

print(f"Found {len(humans)} human cutouts and {len(bgs)} backgrounds.")
print(f"Poacher backgrounds: {len(poacher_bgs)}, Empty backgrounds: {len(empty_bgs)}")

def paste_synthetic(bg_path, human_path=None):
    """
    Pastes a human cutout onto a background image with random transformations 
    to create a synthetic poacher or empty image.
    """
    bg = Image.open(bg_path).convert("RGBA")
    
    if human_path:
        # Load and transform human cutout
        hm = Image.open(human_path).convert("RGBA")
        
        # Random scaling
        scale = random.uniform(0.25, 0.75)
        # Using a safer way to calculate new dimensions to avoid zero size
        new_width = max(1, int(hm.width * scale))
        new_height = max(1, int(hm.height * scale))
        hm = hm.resize((new_width, new_height), Image.LANCZOS)
        
        # Random mirroring
        if random.random() < 0.5: 
            hm = ImageOps.mirror(hm)
        
        # Random alpha (opacity) for blend
        alpha_val = random.uniform(0.6, 1.0)
        alpha = hm.split()[-1].point(lambda p: int(p * alpha_val))
        hm.putalpha(alpha)
        
        # Random position (allowing slight bleed off the edge)
        # This part was corrected from the user's provided code which had syntax errors in calculation
        bleed_x = int(0.15 * bg.width)
        bleed_y = int(0.15 * bg.height)
        x = random.randint(-bleed_x, max(0, bg.width - hm.width + bleed_x))
        y = random.randint(-bleed_y, max(0, bg.height - hm.height + bleed_y))
        
        # Composite the images
        layer = Image.new("RGBA", bg.size)
        layer.paste(hm, (x, y), hm)
        comp = Image.alpha_composite(bg, layer).convert("RGB")
    else:
        # For 'empty' images, just use the background
        comp = bg.convert("RGB")

    # Random enhancements and filters
    if random.random() < 0.6: 
        comp = comp.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.3)))
    
    comp = ImageEnhance.Brightness(comp).enhance(random.uniform(0.75, 1.25))
    comp = ImageEnhance.Contrast(comp).enhance(random.uniform(0.85, 1.15))
    comp = ImageEnhance.Color(comp).enhance(random.uniform(0.85, 1.15))

    return comp

def generate_synthetic_test(count=200):
    """Generates the specified number of synthetic 'poacher' and 'empty' test images."""
    poacher_bgs_copy = list(poacher_bgs)
    empty_bgs_copy = list(empty_bgs)
    
    for i in range(count):
        # Ensure there are backgrounds to pick from, otherwise loop indefinitely
        if not poacher_bgs_copy: 
            poacher_bgs_copy = list(poacher_bgs)
        if not empty_bgs_copy: 
            empty_bgs_copy = list(empty_bgs)
        
        # Poacher
        img = paste_synthetic(random.choice(poacher_bgs_copy), random.choice(humans))
        img.save(test_dir/"poacher"/f"synth_{i}.jpg", quality=90)

        # Empty
        src = random.choice(empty_bgs_copy)
        img_empty = paste_synthetic(src, None)
        img_empty.save(test_dir/"empty"/f"synth_{i}.jpg", quality=90)

    print(f"✅ Generated {count} synthetic test images (Poacher + Empty).")

print("\n--- GENERATING SYNTHETIC TEST DATA ---")
generate_synthetic_test(CONFIG["synthetic_test_count"])
# --- TEST DATA GENERATOR ---
print("\n--- SETTING UP TEST DATA GENERATOR ---")
try:
    # Try to import for EfficientNetV2
    from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
except ImportError:
    # Fallback for EfficientNet (V1)
    from tensorflow.keras.applications.efficientnet import preprocess_input

# Create the data generator (with no augmentation, only preprocessing)
test_aug = ImageDataGenerator(preprocessing_function=preprocess_input)

# Create the test generator from directory
test_gen = test_aug.flow_from_directory(
    test_dir,
    target_size=CONFIG["img_size"],
    batch_size=CONFIG["batch_size"],
    class_mode="categorical",
    shuffle=False # IMPORTANT: Must be False for correct evaluation metrics
)

# --- LOAD TRAINED MODEL ---
print("\n--- LOADING MODEL ---")
model_path = "/kaggle/input/poacher-detection-efficientnetv2/tensorflow2/default/1/final_model.keras"
try:
    model = load_model(model_path)
    print("✅ Model loaded successfully.")
except Exception as e:
    print(f"❌ Error loading model from {model_path}: {e}")
    # Exit or handle error gracefully if model is essential

# --- PREDICTIONS & METRICS ---
print("\n--- MAKING PREDICTIONS ---")
y_true, y_pred, file_paths = [], [], []

# Loop through the generator to get predictions for all batches
for i in range(len(test_gen)):
    x_batch, y_batch = test_gen[i]
    
    # Predict the batch
    preds = model.predict(x_batch, verbose=0)
    
    # Convert one-hot to class index
    y_true.extend(np.argmax(y_batch, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
    
    # Get file paths for later visualization
    # The generator only exposes filepaths for the *current* batch index within the total list
    start_idx = i * test_gen.batch_size
    end_idx = min((i + 1) * test_gen.batch_size, len(test_gen.filepaths))
    file_paths.extend(test_gen.filepaths[start_idx:end_idx])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\n=== Test Metrics ===")
# Overall Accuracy
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

# Classification Report (Precision, Recall, F1-Score)
target_names = list(test_gen.class_indices.keys())
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))
print("--------------------")
# --- RANDOM 20 IMAGE SAMPLES (VISUALIZATION) ---
print("\n--- VISUALIZING 20 RANDOM SAMPLES ---")
sample_idxs = random.sample(range(len(file_paths)), min(20, len(file_paths)))

plt.figure(figsize=(15, 10))

for i, idx in enumerate(sample_idxs):
    # Load the image
    img = load_img(file_paths[idx], target_size=CONFIG["img_size"])
    
    # Setup subplot
    plt.subplot(4, 5, i + 1)
    plt.imshow(img)
    plt.axis('off')
    
    # Get human-readable labels
    class_labels = target_names
    true_label = class_labels[y_true[idx]]
    pred_label = class_labels[y_pred[idx]]
    
    # Set title with color coding for correct/incorrect prediction
    color = 'green' if true_label == pred_label else 'red'
    plt.title(f"T: {true_label}\nP: {pred_label}", fontsize=9, color=color)

plt.suptitle(f"Random Sample Predictions (N={len(sample_idxs)})", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to make room for suptitle
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, precision_score, recall_score, f1_score

# --------------------------------------
# EXTRA METRICS
# --------------------------------------
print("\n--- EXTRA METRICS ---")

# Per-class Precision
prec = precision_score(y_true, y_pred, average=None)
print("Per-class Precision:", prec)

# Per-class Recall
rec = recall_score(y_true, y_pred, average=None)
print("Per-class Recall:", rec)

# Per-class F1
f1 = f1_score(y_true, y_pred, average=None)
print("Per-class F1:", f1)

# Macro & micro
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

# Balanced Accuracy
from sklearn.metrics import balanced_accuracy_score
print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))

# --------------------------------------
# CONFUSION MATRIX PLOT
# --------------------------------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.show()

# --------------------------------------
# ROC CURVE FOR EACH CLASS
# --------------------------------------
# Convert labels to one-hot (needed for ROC)
y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=len(target_names))

y_pred_prob = model.predict(test_gen, verbose=0)

plt.figure(figsize=(8,6))
for i, class_name in enumerate(target_names):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} (AUC = {roc_auc:.2f})")

plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (One-vs-Rest)")
plt.legend()
plt.show()


In [ ]:
# ------------------------------------------------------------
# GRAD-CAM IMPLEMENTATION
# ------------------------------------------------------------
import tensorflow as tf
import numpy as np
import cv2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def find_last_conv(model):
    """Recursively find the last Conv2D layer in any model."""
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    # Search nested models
    for layer in reversed(model.layers):
        if hasattr(layer, "layers"):
            name = find_last_conv(layer)
            if name:
                return name
    return None


def make_gradcam_heatmap(img_array, model, last_conv_name, pred_index=None):
    """Generate Grad-CAM heatmap."""
    # Model to map input → activations of last conv + predictions
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_name).output, model.output]
    )

    # Compute the gradient of the predicted class w.r.t. feature maps
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        pred_score = predictions[:, pred_index]

    grads = tape.gradient(pred_score, conv_outputs)

    # Global average pooling
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(pooled_grads * conv_outputs, axis=-1)

    # Normalize heatmap
    heatmap = np.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(heatmap, image, alpha=0.4):
    """Overlay Grad-CAM heatmap on original image."""
    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap = np.uint8(255 * heatmap)

    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(image, 1 - alpha, heatmap, alpha, 0)
    return overlay
# ------------------------------------------------------------
# VISUALIZE GRAD-CAM FOR RANDOM 20 IMAGES
# ------------------------------------------------------------
print("\n--- GRAD-CAM VISUALIZATION (20 Images) ---")

last_conv_name = find_last_conv(model)
print("Last Conv2D layer:", last_conv_name)

# Pick 20 samples
sample_idxs = random.sample(range(len(file_paths)), min(20, len(file_paths)))

plt.figure(figsize=(15, 25))   # Taller figure

rows, cols = 4, 5   # 4 rows × 5 columns = 20 images

for i, idx in enumerate(sample_idxs):

    img_path = file_paths[idx]

    # Load image for display
    original = load_img(img_path, target_size=CONFIG["img_size"])
    original_arr = img_to_array(original)  
    rgb_display = original_arr.astype("uint8")

    # Preprocess for model
    x = img_to_array(original)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)

    # Grad-CAM heatmap
    heatmap = make_gradcam_heatmap(x, model, last_conv_name)

    # Overlay heatmap
    overlay = overlay_gradcam(heatmap, rgb_display)

    # Prediction labels
    t_label = target_names[int(y_true[idx])]
    p_label = target_names[int(y_pred[idx])]
    color = "green" if t_label == p_label else "red"

    # Plot
    plt.subplot(rows, cols, i+1)
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(f"T: {t_label} | P: {p_label}", color=color, fontsize=9)

plt.tight_layout()
plt.show()
